# 🥇 Gold Layer — First Order Products
**LushProtein Analytics | Medallion Architecture**

Reads from `medallion/silver/` and `medallion/gold/`, extracts line items from each customer's first order, enriches with the product master, and writes to `medallion/gold/`.

| Source Tables | Gold Table | Key Transformations |
|---|---|---|
| `silver_orders`, `silver_products`, `gold_customer_orders`, `gold_customer_profiles` | `gold_first_order_products` | Filter to first order line items, normalise SKU, derive SKU prefix, cast numerics, enrich with product master & order metadata, flag missing SKUs, rename columns |

## 0. Setup & Paths

In [14]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

BASE       = Path.cwd()
SILVER_DIR = BASE / 'medallion' / 'silver'
GOLD_DIR   = BASE / 'medallion' / 'gold'
GOLD_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base   : {BASE}")
print(f"Silver : {SILVER_DIR}")
print(f"Gold   : {GOLD_DIR}")

Base   : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505
Silver : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505/medallion/silver
Gold   : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505/medallion/gold


---
## 1. First Order Products (`silver_orders` + `silver_products` → `gold_first_order_products`)

### Step 1 — Load data & get first order IDs per customer

In [15]:
# Load data 
co  = pd.read_parquet(GOLD_DIR / 'gold_customer_orders.parquet')
df  = pd.read_parquet(SILVER_DIR / 'silver_orders.parquet')
pm  = pd.read_parquet(SILVER_DIR / 'silver_products.parquet')

# Get first order IDs per customer
first_orders = co[co['is_first_order']][['customer_id', 'order_id',
                                          'processed_at', 'country_code',
                                          'channel']].copy()

# Merge repeat purchase flags from gold_customer_profiles
cp = pd.read_parquet(GOLD_DIR / 'gold_customer_profiles.parquet')
first_orders = first_orders.merge(
    cp[['customer_id', 'repeat_purchase_90d', 'days_to_second_order']],
    on='customer_id', how='left'
)

### Step 2 — Filter `silver_orders` to first order line items only

In [16]:
# Filter silver_orders to first order line items only
line_items = df[df['Line: Type'] == 'Line Item'].copy()
first_order_ids = set(first_orders['order_id'])
first_line_items = line_items[line_items['ID'].isin(first_order_ids)].copy()

print(f"First order line items: {len(first_line_items):,}")

First order line items: 26,441


### Step 3 — Normalise SKU & derive SKU prefix (product category)

In [17]:
# Normalise SKU 
first_line_items['Line: Variant SKU'] = (first_line_items['Line: Variant SKU']
                                          .str.strip().str.upper())

# Derive SKU prefix (product category) 
first_line_items['sku_prefix'] = first_line_items['Line: Variant SKU'].str.split('-').str[0]

# Infer product category from Line: Title for rows with no SKU
def infer_category(title):
    if pd.isna(title): return None
    title = title.lower()
    if 'clear' in title:                                                    return 'CLEAR'
    if 'lean' in title:                                                     return 'LEAN'
    if 'plant' in title:                                                    return 'PLT'
    if 'collagen' in title:                                                 return 'COL'
    if 'soy' in title:                                                      return 'SOY'
    if 'creatine' in title:                                                 return 'CRE'
    if 'prime whey' in title:                                               return 'PRI'
    if 'better whey' in title:                                              return 'BET'
    if 'elite' in title:                                                    return 'ELT'
    if any(x in title for x in ['shaker', 'bottle', 'hat', 'shirt', 'band', 'sweater']): return 'ACC'
    if any(x in title for x in ['pack', 'bundle', 'starter']):             return 'BND'
    if any(x in title for x in ['capsule', 'omega', 'l-carnitine', 'green tea']): return 'CAP'
    return None

# Fill sku_prefix where null using title inference
mask = first_line_items['sku_prefix'].isna()
first_line_items.loc[mask, 'sku_prefix'] = first_line_items.loc[mask, 'Line: Title'].apply(infer_category)

# Flag whether category was inferred or from SKU
first_line_items['category_source'] = 'sku'
first_line_items.loc[first_line_items['Line: Variant SKU'].isna(), 'category_source'] = 'inferred'

print(f"\nCategory source breakdown:")
print(first_line_items['category_source'].value_counts().to_string())
print(f"\nCategory breakdown after inference:")
print(first_line_items['sku_prefix'].value_counts(dropna=False).to_string())


Category source breakdown:
category_source
inferred    16916
sku          9525

Category breakdown after inference:
sku_prefix
BET      5902
LEAN     3626
CLEAR    3502
ACC      3190
PLT      2704
BND      1658
COL      1368
CAP      1132
CRE       873
SOY       836
None      824
PRI       754
ELT        72


### Step 4 — Cast numeric columns

In [18]:
# Cast numeric columns
for col in ['Line: Quantity', 'Line: Price', 'Line: Discount', 'Line: Total']:
    first_line_items[col] = pd.to_numeric(first_line_items[col], errors='coerce').round(2)

### Step 5 — Enrich with product master & first order metadata

In [19]:
# Merge with product master
first_line_items = first_line_items.merge(
    pm[['Variant SKU', 'Title', 'Variant Grams']],
    left_on='Line: Variant SKU', right_on='Variant SKU', how='left'
).drop(columns=['Variant SKU'])

# Merge with first order metadata
first_line_items = first_line_items.merge(
    first_orders[['order_id', 'customer_id', 'processed_at', 'country_code',
                  'channel', 'repeat_purchase_90d', 'days_to_second_order']],
    left_on='ID', right_on='order_id', how='left'
).drop(columns=['order_id']) 

### Step 6 — Flag missing SKUs, select & rename final columns, save

In [20]:
# Flag orders with no SKU data
first_line_items['has_sku'] = first_line_items['Line: Variant SKU'].notna()
no_sku_orders = (~first_line_items['has_sku']).sum()
print(f"Line items without SKU: {no_sku_orders:,}")

# Select and rename final columns
gold_first_order_products = first_line_items[[
    'customer_id', 'ID', 'processed_at', 'country_code', 'channel',
    'Line: Variant SKU', 'sku_prefix', 'Line: Title', 'Line: Variant Title',
    'Title', 'Variant Grams',
    'Line: Quantity', 'Line: Price', 'Line: Discount', 'Line: Total',
    'has_sku', 'repeat_purchase_90d', 'days_to_second_order', 'category_source'
]].rename(columns={
    'ID'                  : 'order_id',
    'Line: Variant SKU'   : 'variant_sku',
    'sku_prefix'          : 'product_category',
    'Line: Title'         : 'line_title',
    'Line: Variant Title' : 'variant_title',
    'Title'               : 'product_title',
    'Variant Grams'       : 'variant_grams',
    'Line: Quantity'      : 'quantity',
    'Line: Price'         : 'unit_price',
    'Line: Discount'      : 'line_discount',
    'Line: Total'         : 'line_total',
})

print(f"\ngold_first_order_products: {gold_first_order_products.shape[0]:,} rows × {gold_first_order_products.shape[1]} cols")
print(f"Orders with SKU data: {gold_first_order_products['has_sku'].sum():,}")
print(f"\nProduct category breakdown:\n{gold_first_order_products['product_category'].value_counts(dropna=False).to_string()}")
print(f"\nRepeat purchase 90d rate (SKU orders only):")
sku_only = gold_first_order_products[gold_first_order_products['has_sku']]
print(f"  {sku_only['repeat_purchase_90d'].mean():.1%}")

gold_first_order_products.to_parquet(GOLD_DIR / 'gold_first_order_products.parquet', index=False)
print("\n✅ Saved: gold_first_order_products.parquet")

Line items without SKU: 16,916

gold_first_order_products: 26,441 rows × 19 cols
Orders with SKU data: 9,525

Product category breakdown:
product_category
BET      5902
LEAN     3626
CLEAR    3502
ACC      3190
PLT      2704
BND      1658
COL      1368
CAP      1132
CRE       873
SOY       836
None      824
PRI       754
ELT        72

Repeat purchase 90d rate (SKU orders only):
  19.3%

✅ Saved: gold_first_order_products.parquet


#### Overall
26,441 first order line items across 13,885 customers — on average each customer bought ~1.9 products in their first order.

#### SKU Coverage
- With SKU: 9,525 (36%) — exact SKU known, used for variant-level analysis
- Inferred from title: 16,092 (61%) — product category inferred via keyword matching on `Line: Title`
- Still unclassifiable: 824 (3%) — titles too ambiguous to categorise (likely free gifts or custom bundles)

The original 64% null rate for `Line: Variant SKU` was predominantly driven by Lazada (99.6% of Lazada orders carry no SKU). Category inference from `Line: Title` recovers 61% of these rows at the category level, significantly improving coverage for Q1 analysis.

#### Product Category Breakdown (all first orders, SKU + inferred)

| Category | Count | Notes |
|---|---|---|
| BET | 5,902 | Better Whey — most popular overall, dominant on Lazada MY |
| LEAN | 3,626 | Lean Protein |
| CLEAR | 3,502 | Clear Protein — most popular on DTC |
| ACC | 3,190 | Accessories (shakers, bottles, apparel) |
| PLT | 2,704 | Plant Protein |
| BND | 1,658 | Bundles |
| COL | 1,368 | Collagen |
| CAP | 1,132 | Capsules & supplements |
| CRE | 873 | Creatine |
| SOY | 836 | Soy Protein |
| PRI | 754 | Prime Whey |
| ELT | 72 | Elite Protein |

Note: Rankings differ from SKU-only view — Lazada data shifts BET to #1, reflecting its dominance in the Malaysian marketplace channel. CLEAR and LEAN lead in DTC, connecting to Q4 (Channel Heterogeneity) and Q9 (Geographic Segmentation).

#### 90-day Repeat Purchase Rate (SKU orders only): 19.3%
19.3% of customers with SKU-trackable first orders made a repeat purchase within 90 days — slightly lower than the overall 21.8% in `gold_customer_profiles`, suggesting DTC customers may have marginally lower short-term retention than marketplace customers. This warrants further investigation in Q4.